# Merge Entries with Claims

After the work I've done to extract data from TEI XML and convert it to JSON, I need to merge the apparatus entries with their apparatus claims. I confess that I needed to consult ChatGPT for help with this.

In [29]:
import json
from pathlib import Path
from collections import defaultdict

First I should remove all the apparatus_claims that hold no information—a leftover from a previous operation.

In [9]:
with open("../converted-json/calpSicPoem1-target-array.json", "r") as f:
    data = json.load(f)
    claims = [claim for claim in data]
    original_count = len(claims)

    claims = [
        claim
        for claim in claims
        if not (
            claim.get("reading_type") == "lemma" and claim.get("supporting_evidence") == []
        )
    ]

    print(f"Removed {original_count - len(claims)} erroneous claims.")

Removed 169 erroneous claims.


In [30]:
with open("../converted-json/apparatus_claims.json", "w") as f:
    json.dump(claims, f, indent=4, ensure_ascii=False)

In [31]:
# ChatGPT wrote this function to merge apparatus entries with their associated claims.
def merge_apparatus(entries_file, claims_file, output_file):
    """
    Merge apparatus entries with their associated claims.
    
    Args:
    - entries_file: Path to the JSON file containing apparatus entries.
    - claims_file: Path to the JSON file containing claims.
    - output_file: Path to the output JSON file where the merged data will be saved.
    
    Returns:
    - A list containing the merged apparatus entries and claims.
    """

    # Load both JSON files
    with Path(entries_file).open("r", encoding="utf-8") as f:
        entries = json.load(f)

    with Path(claims_file).open("r", encoding="utf-8") as f:
        claims = json.load(f)

    # Group claims by apparatus_entry_id
    claims_by_entry = defaultdict(list)

    for claim in claims:
        claims_by_entry[claim["apparatus_entry_id"]].append(claim)

    # Assemble the combined list
    merged = []

    for entry in entries:
        merged.append(entry)

        # Retrieve claims associated with this entry
        associated_claims = claims_by_entry.get(entry["id"], [])

        # Sort claims by reporting_rank
        associated_claims.sort(
            key=lambda claim: claim.get("reporting_rank") or float("inf")
        )

        merged.extend(associated_claims)

    entry_ids = {entry["id"] for entry in entries}

    orphaned_claims = [
        claim for claim in claims if claim["apparatus_entry_id"] not in entry_ids
    ]

    if orphaned_claims:
        for claim in orphaned_claims:
            print(
                f"Orphaned claim found: {claim['id']} with cts_id {claim['target']} and apparatus_entry_id {claim['apparatus_entry_id']}"
            )
        raise ValueError(f"{len(orphaned_claims)} claims have no matching apparatus entry.")

    # Write the merged JSON
    with Path(output_file).open("w", encoding="utf-8") as f:
        json.dump(merged, f, indent=4, ensure_ascii=False)

    return merged

In [34]:
entries = "../converted-json/apparatus_entries.json"
claims_file = "../converted-json/apparatus_claims.json"
output_file = "../converted-json/merged-apparatus-entries-and-claims.json"

In [36]:
merged = merge_apparatus(entries, claims_file, output_file)

In [37]:
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=4, ensure_ascii=False)

Validation time!

- No duplicated `id` values
- Every claim points to a valid `apparatus_entry_id`
- No orphaned apparatus entries or claims
- Every claim should have a value in `supporting_evidence`

In [38]:
with open(output_file, "r", encoding="utf-8") as f:
    merged_data = json.load(f)

In [45]:
# Check for duplicated "id" values in the merged data
ids = [item["id"] for item in merged_data]
if len(ids) != len(set(ids)):
    duplicates = set([id for id in ids if ids.count(id) > 1])
    raise ValueError(f"Duplicate 'id' values found in merged data: {duplicates}")

In [47]:
# Every claim should point to a corresponding apparatus entry
entry_ids = {
    entry["id"] for entry in merged_data if entry.get("type") == "apparatus_entry"
}
claim_targets = {
    claim["apparatus_entry_id"] for claim in merged_data if claim.get("type") == "apparatus_claim"
}
for claim_target in claim_targets:
    if claim_target not in entry_ids:
        raise ValueError(f"Claim points to non-existent apparatus entry: {claim_target}")

In [ ]:
# Make sure that all claims have a corresponding apparatus entry
orphaned_claims = [
    claim for claim in merged_data if claim.get("type") == "apparatus_claim" and claim["apparatus_entry_id"] not in entry_ids
]
if orphaned_claims:
    raise ValueError(f"Orphaned claims found: {orphaned_claims}")

In [48]:
# Every claim should have a value in the "supporting_evidence" field
no_evidence_claims = [
    claim for claim in merged_data if claim.get("type") == "apparatus_claim" and not claim.get("supporting_evidence")
]
if no_evidence_claims:
    raise ValueError(f"Claims without supporting evidence found: {no_evidence_claims}")

ValueError: Claims without supporting evidence found: [{'type': 'apparatus_claim', 'id': 'urn:uuid:28cd329f-efdc-4f50-8e1c-f39111bf32a8', 'apparatus_entry_id': 'urn:uuid:1bf943f0-c283-40f1-9a55-4c684199c699', 'target': ['urn:cts:latinLit:phi0830.phi001.dll:1.19@O.'], 'claim_type': 'reading', 'reading_type': 'variant', 'reading': None, 'supporting_evidence': [], 'reporting_rank': 1, 'editorial_comments': 'rell. codd. et edd.', 'confidence': {'interpretation': None, 'reasons': []}, 'evaluation': {'target_correct': None, 'reading_correct': None, 'witnesses_correct': None, 'interpretation_correct': None, 'reviewer': None, 'review_status': None}}, {'type': 'apparatus_claim', 'id': 'urn:uuid:515c517c-c5b3-43b1-840f-03c37bff241f', 'apparatus_entry_id': 'urn:uuid:935669b5-28b5-4fb5-8926-fc789fbda482', 'target': ['urn:cts:latinLit:phi0830.phi001.dll:1.25@citius percurrere versus'], 'claim_type': 'reading', 'reading_type': 'variant', 'reading': 'percurrere certius orsus', 'supporting_evidence': 

In [49]:
for claim in no_evidence_claims:
    print(claim)

{'type': 'apparatus_claim', 'id': 'urn:uuid:28cd329f-efdc-4f50-8e1c-f39111bf32a8', 'apparatus_entry_id': 'urn:uuid:1bf943f0-c283-40f1-9a55-4c684199c699', 'target': ['urn:cts:latinLit:phi0830.phi001.dll:1.19@O.'], 'claim_type': 'reading', 'reading_type': 'variant', 'reading': None, 'supporting_evidence': [], 'reporting_rank': 1, 'editorial_comments': 'rell. codd. et edd.', 'confidence': {'interpretation': None, 'reasons': []}, 'evaluation': {'target_correct': None, 'reading_correct': None, 'witnesses_correct': None, 'interpretation_correct': None, 'reviewer': None, 'review_status': None}}
{'type': 'apparatus_claim', 'id': 'urn:uuid:515c517c-c5b3-43b1-840f-03c37bff241f', 'apparatus_entry_id': 'urn:uuid:935669b5-28b5-4fb5-8926-fc789fbda482', 'target': ['urn:cts:latinLit:phi0830.phi001.dll:1.25@citius percurrere versus'], 'claim_type': 'reading', 'reading_type': 'variant', 'reading': 'percurrere certius orsus', 'supporting_evidence': [], 'reporting_rank': 1, 'editorial_comments': None, 'co